# 04. 10-Gesture Benchmark (Direct Literature Comparison)
**Objective:** Benchmark the Hybrid Dictionary Learning pipeline on a 10-gesture subset to establish a direct comparison with Molina et al. (2025).

In [1]:
%load_ext autoreload
%autoreload 2

import sys
import os
import h5py
import numpy as np
import joblib
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, f1_score, classification_report

sys.path.append(os.path.abspath('../'))
from src.config import PREPROCESSED_DIR, MODELS_DIR, MOVEMENT_LABELS
from src.features import extract_fft_magnitude, EMGDictionaryLearner, extract_time_domain_features

### 1. Load Dataset and Filter to 10 Gestures

In [2]:
data_path = os.path.join(PREPROCESSED_DIR, "DB1_S1_Dense.h5")
with h5py.File(data_path, 'r') as f:
    X_all = np.array(f['X'])
    y_all = np.array(f['y']).astype(np.int64)
    reps_all = np.array(f['reps'])

# Select 10 basic gestures (Classes 1 to 10)
selected_gestures = list(range(1, 11))
mask_10g = np.isin(y_all, selected_gestures)

X_10g = X_all[mask_10g]
y_10g = y_all[mask_10g]
reps_10g = reps_all[mask_10g]

train_reps = [1, 3, 4, 6, 7, 8, 9, 10]
test_reps = [2, 5]

train_idx = np.where(np.isin(reps_10g, train_reps))[0]
test_idx = np.where(np.isin(reps_10g, test_reps))[0]

X_train, y_train = X_10g[train_idx], y_10g[train_idx]
X_test, y_test = X_10g[test_idx], y_10g[test_idx]

print(f"10-Gesture Training Samples: {X_train.shape[0]}")
print(f"10-Gesture Testing Samples:  {X_test.shape[0]}")

10-Gesture Training Samples: 25434
10-Gesture Testing Samples:  6541


### 2. Extract Hybrid Features (DL Atoms + Time Domain)

In [3]:
# Frequency feature extraction
X_train_freq = extract_fft_magnitude(X_train)
X_test_freq = extract_fft_magnitude(X_test)

# Train a dedicated 10-gesture dictionary
dl_10g = EMGDictionaryLearner(n_atoms=64, n_nonzero_coefs=5, random_state=42)
dl_10g.fit(X_train_freq)

X_train_sparse = dl_10g.transform(X_train_freq)
X_test_sparse = dl_10g.transform(X_test_freq)

# Time-domain feature extraction
X_train_td = extract_time_domain_features(X_train)
X_test_td = extract_time_domain_features(X_test)

# Feature fusion (64 sparse atoms + 20 MAV/RMS)
X_train_hybrid = np.hstack((X_train_sparse, X_train_td))
X_test_hybrid = np.hstack((X_test_sparse, X_test_td))

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_hybrid)
X_test_scaled = scaler.transform(X_test_hybrid)

print(f"Feature matrix shape: {X_train_scaled.shape}")

Extracting FFT magnitudes...
Extracting FFT magnitudes...
Fitting Dictionary (64 atoms) on shape (25434, 110)...
Dictionary learning complete.
Extracting Time-Domain features (MAV, RMS)...
Extracting Time-Domain features (MAV, RMS)...
Feature matrix shape: (25434, 84)


### 3. Train and Evaluate Classifiers (Random Forest & Non-linear SVM)

In [4]:
# A. Random Forest Classifier
rf_10g = RandomForestClassifier(n_estimators=200, max_depth=20, n_jobs=-1, random_state=42)
rf_10g.fit(X_train_scaled, y_train)
rf_preds = rf_10g.predict(X_test_scaled)

rf_acc = accuracy_score(y_test, rf_preds)
rf_f1 = f1_score(y_test, rf_preds, average='macro')

# B. Non-Linear RBF Support Vector Machine
svm_rbf = SVC(C=10.0, kernel='rbf', gamma='scale', random_state=42)
svm_rbf.fit(X_train_scaled, y_train)
svm_preds = svm_rbf.predict(X_test_scaled)

svm_acc = accuracy_score(y_test, svm_preds)
svm_f1 = f1_score(y_test, svm_preds, average='macro')

print("\n" + "=" * 55)
print(" 10-GESTURE BENCHMARK EVALUATION (SUBJECT 1)")
print("=" * 55)
print(f"Random Forest (10G)  | Accuracy: {rf_acc * 100:.2f}% | Macro F1: {rf_f1 * 100:.2f}%")
print(f"RBF-SVM       (10G)  | Accuracy: {svm_acc * 100:.2f}% | Macro F1: {svm_f1 * 100:.2f}%")
print("=" * 55)


 10-GESTURE BENCHMARK EVALUATION (SUBJECT 1)
Random Forest (10G)  | Accuracy: 76.95% | Macro F1: 77.56%
RBF-SVM       (10G)  | Accuracy: 70.69% | Macro F1: 71.53%
